# Solutions · 00 · Randomness and pseudo-random generators

Worked solutions to the exercises of [notebook 00](../notebooks/00_randomness_and_generators.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engstoch import rng as rg, rngtests as rt, special as sf
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

Find all multipliers a in 1..63 for which x -> a x + 1 mod 64 has full period, and check the Hull-Dobell theorem against `LCG.period` for every a.

In [2]:
full = [a for a in range(1, 64) if rg.LCG(a, 1, 64).period() == 64]
hd = [a for a in range(1, 64) if rg.full_period_conditions(a, 1, 64)]
print("full period (by running the generator):", full)
print("Hull-Dobell prediction agrees:", full == hd, "-> a = 1 mod 4:", all(a % 4 == 1 for a in full), f"({len(full)} multipliers)")

full period (by running the generator): [1, 5, 9, 13, 17, 21, 25, 29, 33, 37, 41, 45, 49, 53, 57, 61]
Hull-Dobell prediction agrees: True -> a = 1 mod 4: True (16 multipliers)


With m = 64 = 2⁶ and c = 1 (odd, so coprime to m), the theorem requires a − 1 divisible by 2 (the only prime
factor) and by 4 (because 4 divides m): a ≡ 1 (mod 4). Exactly those 16 multipliers give period 64 - though
a = 1, the identity plus a shift, shows that full period alone says nothing about randomness.

## Exercise 2

Generate 10^6 triples from Park-Miller and from RANDU and count how many fall in the slab 0.45 < 9u_1 - 6u_2 + u_3 - round(9u_1 - 6u_2 + u_3) < 0.55. What fraction does a perfect generator give, and what do the two generators give?

In [3]:
rows = []
for name, gen in (("Park-Miller", rg.LCG.park_miller(7)), ("RANDU", rg.LCG.randu(7))):
    u = gen.random(3 * 10**6).reshape(-1, 3)
    s = 9 * u[:, 0] - 6 * u[:, 1] + u[:, 2]
    frac = s - np.round(s)
    rows.append((name, float(np.mean((frac > 0.45) & (frac < 0.55)) + np.mean((frac < -0.45) & (frac > -0.55)))))
print(pd.DataFrame(rows, columns=["generator", "fraction in the slab |frac| in (0.45, 0.55)"]).to_string(index=False))
v = R(1).random((10**6, 3)); s = 9 * v[:, 0] - 6 * v[:, 1] + v[:, 2]; f = s - np.round(s)
print("a perfect generator (NumPy PCG64):", np.mean(np.abs(f) > 0.45).round(4), "(0.1: the fractional part of a sum of continuous variables is ~uniform)")

  generator  fraction in the slab |frac| in (0.45, 0.55)
Park-Miller                                     0.100108
      RANDU                                     0.000000
a perfect generator (NumPy PCG64): 0.1003 (0.1: the fractional part of a sum of continuous variables is ~uniform)


For a good generator the fractional part of 9u₁ − 6u₂ + u₃ is essentially uniform, so a slab of width 0.1 (the
two halves near ±½) holds 10 % of the triples - Park-Miller gives that. For RANDU the combination is *exactly*
an integer (up to rounding), the fractional part is 0, and the slab is empty: a test aimed at the right
direction exposes the defect with a handful of numbers.

## Exercise 3

**Implement it yourself:** write Knuth's gap test for the interval [0, 0.1) from scratch (gap lengths 0..9 and >= 10, expected counts from the geometric law, the chi-square p-value from `special.chi2_sf`) and check it against `rngtests.gap_test` on 10^5 PCG32 numbers.

In [4]:
def gap_test_mine(u, a=0.0, b=0.1, t=10):
    hits = np.flatnonzero((u >= a) & (u < b))
    gaps = np.diff(hits) - 1
    p = b - a
    counts = np.bincount(np.minimum(gaps, t), minlength=t + 1)
    probs = np.r_[p * (1 - p) ** np.arange(t), (1 - p) ** t]
    expected = probs * len(gaps)
    stat = np.sum((counts - expected) ** 2 / expected)
    return stat, sf.chi2_sf(stat, t)
u = rg.PCG32(3, 9).random(100_000)
mine = gap_test_mine(u)
lib = rt.gap_test(u, 0.0, 0.1, 10)
print(f"mine: chi2 = {mine[0]:.4f}, p = {mine[1]:.4f};  library: chi2 = {lib['statistic']:.4f}, p = {lib['p_value']:.4f}")

mine: chi2 = 7.0289, p = 0.7227;  library: chi2 = 7.0289, p = 0.7227


The gaps between visits to an interval of probability p are geometric: P(gap = r) = p(1 − p)ʳ. Pooling the tail
at t keeps every expected count large enough for the chi-square approximation (t + 1 cells, t degrees of
freedom). The hand-written test reproduces the library's statistic exactly.